# A different deployment question: exporting to ExecuTorch

**You are here → 05 — ExecuTorch portability.**

**Main workflow:** serving foundations → Qwen optimization → Nemotron optimization.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Serving foundations | [00 — Serving foundations](00_local_serving.ipynb) | Follow the clinical workload, schema context, local serving and SQL evaluation using recorded comparisons. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test hybrid caching, context reduction, concurrency, scheduler capacity, reasoning and output limits. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Inspect packaged deployment, profiles and request contracts, including a failed smoke and its correction. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| **→ Optional CPU portability (current)** | [05 — ExecuTorch portability](05_edge_portability_executorch.ipynb) | Inspect the recorded SmolLM2 CPU export/run, separate from clinical-model performance comparisons. |

So far we've sent requests to an inference server. What if we want a model
inside an application instead? This optional branch follows an existing
export-and-run experiment with ExecuTorch on the M4 Pro CPU.

We used **SmolLM2-135M**, a small base completion model, rather than the 4B
clinical models. It let us check conversion, export and native execution;
it does not establish clinical accuracy or a speed ranking against Qwen or Nemotron.
The earlier Llama 3.2 1B plan lacked checkpoint access, so the completed example
used the publicly accessible SmolLM2 checkpoint.

Choose **Run All** with the [CPU notebook environment](../SETUP.md) to inspect
the saved conversion checks, timings, footprint and generated sample. This is
a read-only replay: it does not install ExecuTorch/PyTorch, download a model,
or rerun export, regardless of other chapters' Replay/Plan/Live settings.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.edge_notebook import recorded_edge
edge = recorded_edge(ROOT)
print(edge['label'])
display(JSON({k: edge[k] for k in ('source', 'source_sha256', 'model_name', 'model_revision',
    'model_sha256', 'tokenizer_sha256', 'runner_sha256', 'backend', 'hardware', 'dtype',
    'max_context_length', 'threads', 'versions')}))


recorded Mac CPU portability measurement — October 29, 2026


<IPython.core.display.JSON object>

## 1. Follow the model from checkpoint to native execution

The recorded path was:

**Pinned HF checkpoint → FP32 weight conversion → reference-logit checks
→ XNNPACK `.pte` export → native C++ TextLLMRunner through Python bindings.**

The exporter reused the upstream `Llama2Model` architecture class. That class
name does not turn the checkpoint into Llama—the model is still SmolLM2-135M.
Before export, two input sequences were checked against the original HF model
with `atol=rtol=0.001`. Run the next cell to inspect the recorded validation.
Those checks support the conversion, not downstream task accuracy.

The native runner also needed custom-kernel registration before loading the
program. The [historical recipe](../baseline/mvp3/tutorial/EXECUTORCH_MAC.md)
describes that work, but its old script paths are not a supported fresh-clone
execution recipe. See [the current scope](../tutorial/EXECUTORCH_MAC.md).


In [2]:
display(JSON(edge['checkpoint_validation']))


<IPython.core.display.JSON object>

## 2. What do the timing and size numbers actually include?

Before reading the table, separate the boundaries:

- **Model load** excludes Python initialization.
- **Single request** includes processing the prompt and generating output.
- **Whole-process time** also includes child-process startup and runtime initialization.
- **Artifact size** includes the `.pte` and tokenizer, not the installed Python/PyTorch libraries.
- **Process RSS** includes loaded libraries; it is not VRAM or just the model weights.

The next cell shows these measurements and the generated sample. Sampled
memory can miss brief peaks, and runtime operator fallback has not been reviewed.


In [3]:
rows = [
    ('Native model load', f"{edge['startup_seconds']:.3f} s"),
    ('Native single request, including prompt evaluation', f"{edge['single_request_seconds']:.3f} s"),
    ('Whole child process', f"{edge['wall_seconds']:.3f} s"),
    ('PTE + tokenizer footprint', f"{edge['footprint_bytes'] / 2**20:.2f} MiB"),
    ('Sampled peak process RSS', f"{edge['memory']['peak_bytes'] / 2**20:.2f} MiB"),
    ('Native prompt/generated tokens', f"{edge['generation']['num_prompt_tokens']} / {edge['generation']['num_generated_tokens']}"),
    ('Reviewed operator fallback', 'unavailable' if edge['operator_fallback_count'] is None else str(edge['operator_fallback_count'])),
]
display(Markdown('| Metric | Recorded Mac CPU value |\n|---|---|\n' +
                 '\n'.join(f'| {name} | {value} |' for name, value in rows)))
display(JSON({'prompt': edge['generation']['prompt'], 'generated_text': edge['generation']['text'],
              'limitations': edge['limitations']}))


| Metric | Recorded Mac CPU value |
|---|---|
| Native model load | 0.076 s |
| Native single request, including prompt evaluation | 0.199 s |
| Whole child process | 3.480 s |
| PTE + tokenizer footprint | 623.60 MiB |
| Sampled peak process RSS | 1018.12 MiB |
| Native prompt/generated tokens | 4 / 31 |
| Reviewed operator fallback | unavailable |

<IPython.core.display.JSON object>

## 3. What did we learn from this portability check?

The saved CPU/XNNPACK run loaded the model in **0.076 s** and handled its one
sample request in **0.199 s**, including prompt processing. The whole child
process took **3.480 s**. Model plus tokenizer occupied **623.60 MiB**, while
sampled peak process RSS was **1018.13 MiB**.

The difference between request time and process time is the lesson: an embedded
deployment has startup and library costs as well as inference costs. Likewise,
an export file's size is not the application's complete memory footprint.

This was one smoke sample with a warm filesystem, FP32 weights, and a small
completion model. It was not a concurrent-server or clinical-task benchmark,
and export delegation does not prove which operators fell back at runtime.
Before choosing an edge model, evaluate its intended task and inspect backend
execution separately.

Return to [the deployment decision guide](../tutorial/RESULTS.md#what-should-i-choose),
revisit [Qwen optimization](01_qwen_cache_quantization.ipynb), or inspect the
[original Mac measurement](../experiment-tracker/tutorial/edge/20261029T161510_034051Z.json).
